In [1]:
! pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00


In [2]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

import torch

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import bitsandbytes as bnb

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [6]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files='/content/tech_knowledge_dataset.csv')

print(dataset)
print(dataset['train'].column_names)
print(dataset['train'][0])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 50
    })
})
['text']
{'text': 'Cloud computing provides on-demand access to computing resources over the internet.'}


In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="tech_knowledge_dataset.csv"
)["train"]

In [8]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [10]:
from transformers import Trainer, TrainingArguments

def add_labels_to_dataset(examples):
    examples["labels"] = examples["input_ids"]
    return examples

tokenized_dataset = tokenized_dataset.map(
    add_labels_to_dataset,
    batched=True
)

training_args = TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
10,5.226963
20,4.775810


TrainOutput(global_step=26, training_loss=4.7626028794508715, metrics={'train_runtime': 225.4718, 'train_samples_per_second': 0.444, 'train_steps_per_second': 0.115, 'total_flos': 1272592937779200.0, 'train_loss': 4.7626028794508715, 'epoch': 2.0})

In [20]:
def generate_response(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

print(generate_response("What is cloud computing?"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is cloud computing?


In [12]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 50
})


In [13]:
print(dataset.column_names)

['text']


In [14]:
print(dataset[0])

{'text': 'Cloud computing provides on-demand access to computing resources over the internet.'}


In [15]:
print(tokenized_dataset)

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 50
})


In [16]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [17]:
trainer.train()

Step,Training Loss
10,3.399724
20,2.303370


TrainOutput(global_step=26, training_loss=2.5659557489248424, metrics={'train_runtime': 231.7497, 'train_samples_per_second': 0.431, 'train_steps_per_second': 0.112, 'total_flos': 1272592937779200.0, 'train_loss': 2.5659557489248424, 'epoch': 2.0})

In [21]:
print(generate_response("What is Google Cloud Platform used for?"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is Google Cloud Platform used for?


In [22]:
print(generate_response("What is blockchain technology?"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is blockchain technology?
